# Diff3F Semantic Feature Exploration for Strata Weight Prediction

**Issue:** [#52](https://github.com/TWoolff/strata-training-data/issues/52)  
**PRD Reference:** Research PRD §8 — Diff3F Semantic Feature Exploration  
**Goal:** Evaluate whether Diffusion 3D Features (Diff3F) could improve Strata's weight painting prediction.

## Background

The [ASMR paper](https://arxiv.org/abs/2503.13579) (Hong et al., 2025) uses Diff3F — semantic
descriptors extracted from diffusion model activations — to improve automatic rigging and
skinning quality. A pixel at an elbow carries features that say "this is a bend region"
regardless of art style.

**Diff3F** ([Dutt et al., CVPR 2024](https://arxiv.org/abs/2311.17024)) extracts per-vertex
semantic features from 3D meshes by:
1. Rendering the mesh from ~100 viewpoints
2. Extracting diffusion features (1280-dim) via ControlNet inpainting
3. Extracting DINO features (768-dim) from the same renders
4. Unprojecting and fusing features to produce 2048-dim per-vertex descriptors

**Strata's current approach:** Vertex weights are extracted from bone-to-region mappings
using geometric position + segmentation label (`pipeline/weight_extractor.py`). Diff3F
features could provide additional semantic signal for weight painting on unusual body
proportions.

## Notebook Structure

1. **Setup & Installation** — Dependencies, model loading
2. **Feature Extraction** — Extract Diff3F features from Strata FBX characters
3. **Visualization** — Clustering, body part grouping, joint proximity analysis
4. **Evaluation** — Quantitative comparison vs. current approach
5. **Conclusion** — Recommendation: integrate or skip

---
## 1. Setup & Installation

### Requirements
- Python 3.10+
- CUDA-capable GPU (16GB+ VRAM recommended)
- ~10GB disk for model weights (Stable Diffusion + DINO)

### Install Diff3F dependencies
```bash
pip install torch torchvision
pip install pytorch3d  # See https://github.com/facebookresearch/pytorch3d/blob/main/INSTALL.md
pip install diffusers transformers accelerate
pip install trimesh numpy scipy scikit-learn matplotlib seaborn
pip install Pillow opencv-python
```

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent.parent  # docs/research/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Verify key dependencies
REQUIRED = ["torch", "numpy", "trimesh", "sklearn", "matplotlib", "PIL"]
OPTIONAL = ["pytorch3d", "diffusers", "transformers"]

for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: MISSING (required)")

print()
for pkg in OPTIONAL:
    try:
        importlib.import_module(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: MISSING (needed for full Diff3F extraction)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import trimesh

# Strata region definitions (mirror of pipeline/config.py, no bpy dependency)
REGION_NAMES: dict[int, str] = {
    0: "background",
    1: "head",
    2: "neck",
    3: "chest",
    4: "spine",
    5: "hips",
    6: "upper_arm_l",
    7: "lower_arm_l",
    8: "hand_l",
    9: "upper_arm_r",
    10: "lower_arm_r",
    11: "hand_r",
    12: "upper_leg_l",
    13: "lower_leg_l",
    14: "foot_l",
    15: "upper_leg_r",
    16: "lower_leg_r",
    17: "foot_r",
    18: "shoulder_l",
    19: "shoulder_r",
}

# Joint body-part groupings for analysis
BODY_GROUPS: dict[str, list[int]] = {
    "head_neck": [1, 2],
    "torso": [3, 4, 5],
    "left_arm": [6, 7, 8, 18],
    "right_arm": [9, 10, 11, 19],
    "left_leg": [12, 13, 14],
    "right_leg": [15, 16, 17],
}

# Region colors for visualization (distinct per region)
REGION_CMAP = plt.cm.get_cmap("tab20", 20)

print(f"Strata skeleton: {len(REGION_NAMES) - 1} body regions + background")
print(f"Body groups: {list(BODY_GROUPS.keys())}")

### 1.1 Load sample meshes

Load a few FBX characters from `data/fbx/` using trimesh. We extract vertex positions
and (if available) bone weight assignments for comparison.

In [ ]:
FBX_DIR = PROJECT_ROOT / "data" / "fbx"


def load_mesh(fbx_path: Path) -> trimesh.Trimesh | None:
    """Load an FBX file and return the combined mesh."""
    try:
        scene = trimesh.load(str(fbx_path), force="scene")
        if isinstance(scene, trimesh.Scene):
            meshes = [g for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh)]
            if not meshes:
                return None
            return trimesh.util.concatenate(meshes)
        return scene
    except Exception as e:
        print(f"  Failed to load {fbx_path.name}: {e}")
        return None


# Find available FBX files
if FBX_DIR.exists():
    fbx_files = sorted(FBX_DIR.glob("*.fbx"))[:5]  # Start with 5 characters
    print(f"Found {len(list(FBX_DIR.glob('*.fbx')))} FBX files, loading first 5...")
else:
    fbx_files = []
    print(f"FBX directory not found: {FBX_DIR}")
    print("Place FBX characters in data/fbx/ to run this notebook.")

sample_meshes: dict[str, trimesh.Trimesh] = {}
for fbx_path in fbx_files:
    mesh = load_mesh(fbx_path)
    if mesh is not None:
        sample_meshes[fbx_path.stem] = mesh
        print(f"  {fbx_path.stem}: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

print(f"\nLoaded {len(sample_meshes)} meshes for analysis.")

---
## 2. Feature Extraction

### 2.1 Diff3F feature extraction (GPU required)

Uses the [Diff3F repository](https://github.com/niladridutt/Diffusion-3D-Features) to
extract per-vertex semantic features. This requires:
- CUDA GPU with 16GB+ VRAM
- Stable Diffusion + ControlNet weights (~5GB)
- DINO ViT-B/8 weights (~350MB)

The extraction renders each mesh from ~100 viewpoints and fuses diffusion + DINO
features per vertex, producing a 2048-dimensional descriptor per vertex.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Diff3F extraction will be very slow.")
    print("Consider using pre-computed features or the DINO-only fallback (Section 2.2).")

In [ ]:
# Clone Diff3F if not already present
DIFF3F_DIR = PROJECT_ROOT / "deps" / "Diffusion-3D-Features"

if not DIFF3F_DIR.exists():
    print("Cloning Diff3F repository...")
    import subprocess

    DIFF3F_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/niladridutt/Diffusion-3D-Features.git",
            str(DIFF3F_DIR),
        ],
        check=True,
    )
    print("Done.")
else:
    print(f"Diff3F already cloned at {DIFF3F_DIR}")

# Add to path
if str(DIFF3F_DIR) not in sys.path:
    sys.path.insert(0, str(DIFF3F_DIR))

In [ ]:
def extract_diff3f_features(
    mesh: trimesh.Trimesh,
    name: str,
    num_views: int = 50,
    resolution: int = 512,
) -> np.ndarray | None:
    """Extract Diff3F per-vertex features from a mesh.

    Args:
        mesh: Input trimesh mesh.
        name: Character name (for logging).
        num_views: Number of render viewpoints (fewer = faster, less accurate).
        resolution: Render resolution.

    Returns:
        Array of shape (num_vertices, 2048) or None if extraction fails.
    """
    try:
        from diff3f import get_features_per_vertex
        from diffusion import load_pipeline
        from dino import load_dino_model
    except ImportError:
        print("Diff3F modules not available. Install dependencies or clone the repo.")
        return None

    print(f"Extracting features for {name} ({len(mesh.vertices)} vertices, {num_views} views)...")

    # Load models (cached after first call)
    if not hasattr(extract_diff3f_features, "_pipe"):
        print("  Loading Stable Diffusion + ControlNet pipeline...")
        extract_diff3f_features._pipe = load_pipeline(DEVICE)
        print("  Loading DINO ViT-B/8...")
        extract_diff3f_features._dino = load_dino_model(DEVICE)

    # Convert trimesh to pytorch3d mesh format
    import pytorch3d.structures

    verts = torch.tensor(mesh.vertices, dtype=torch.float32, device=DEVICE)
    faces = torch.tensor(mesh.faces, dtype=torch.int64, device=DEVICE)
    py3d_mesh = pytorch3d.structures.Meshes(verts=[verts], faces=[faces])

    # Extract features
    features = get_features_per_vertex(
        device=DEVICE,
        pipe=extract_diff3f_features._pipe,
        dino_model=extract_diff3f_features._dino,
        mesh=py3d_mesh,
        prompt="a 3D character model",
        num_views=num_views,
        H=resolution,
        W=resolution,
    )

    features_np = features.cpu().numpy()
    print(f"  Extracted: {features_np.shape} (vertices x feature_dim)")
    return features_np

In [ ]:
# Extract features for all sample meshes
CACHE_DIR = PROJECT_ROOT / "output" / "research" / "diff3f_features"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

vertex_features: dict[str, np.ndarray] = {}

for name, mesh in sample_meshes.items():
    cache_path = CACHE_DIR / f"{name}_diff3f.npy"

    if cache_path.exists():
        print(f"Loading cached features for {name}...")
        vertex_features[name] = np.load(str(cache_path))
    else:
        features = extract_diff3f_features(mesh, name)
        if features is not None:
            vertex_features[name] = features
            np.save(str(cache_path), features)
            print(f"  Cached to {cache_path}")

print(f"\nFeatures extracted for {len(vertex_features)}/{len(sample_meshes)} meshes.")

### 2.2 DINO-only fallback (lightweight, 2D images)

If full Diff3F extraction is unavailable (no GPU / no PyTorch3D), we can extract
DINO features directly from 2D rendered images. This is also the more relevant
path for Strata since weight prediction at inference time operates on 2D images,
not 3D meshes.

DINO ViT features carry strong semantic information about body parts and have been
shown to produce meaningful correspondences across different visual styles.

In [ ]:
from PIL import Image


def extract_dino_features_2d(
    image: Image.Image | np.ndarray,
    model_name: str = "facebook/dinov2-base",
    patch_size: int = 14,
) -> np.ndarray:
    """Extract DINOv2 patch-level features from a 2D image.

    Args:
        image: Input image (PIL or numpy array).
        model_name: HuggingFace model identifier.
        patch_size: ViT patch size (14 for DINOv2 base).

    Returns:
        Array of shape (num_patches_h, num_patches_w, feature_dim).
    """
    from transformers import AutoImageProcessor, AutoModel

    # Load model (cached)
    if not hasattr(extract_dino_features_2d, "_model"):
        print(f"Loading {model_name}...")
        extract_dino_features_2d._processor = AutoImageProcessor.from_pretrained(model_name)
        extract_dino_features_2d._model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    processor = extract_dino_features_2d._processor
    model = extract_dino_features_2d._model

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get patch tokens (exclude CLS token)
    patch_tokens = outputs.last_hidden_state[:, 1:, :]  # (1, num_patches, dim)
    patch_tokens = patch_tokens.squeeze(0).cpu().numpy()

    # Reshape to spatial grid
    input_size = processor.size["height"] if isinstance(processor.size, dict) else 518
    grid_size = input_size // patch_size
    patch_tokens = patch_tokens[: grid_size * grid_size].reshape(grid_size, grid_size, -1)

    return patch_tokens


# Test with a sample rendering (if available)
SEGMENTATION_DIR = PROJECT_ROOT / "output" / "segmentation"
sample_images = list(SEGMENTATION_DIR.rglob("image.png"))[:5] if SEGMENTATION_DIR.exists() else []

if sample_images:
    print(f"Found {len(sample_images)} rendered images for DINO feature extraction.")
else:
    print("No pre-rendered images found in output/segmentation/.")
    print("Run the Blender pipeline first, or use synthetic test images below.")

In [ ]:
# Generate simple test renders from trimesh if no pipeline output exists
test_renders: dict[str, np.ndarray] = {}

if not sample_images and sample_meshes:
    print("Generating test renders from loaded meshes...")
    for name, mesh in list(sample_meshes.items())[:3]:
        try:
            scene = trimesh.Scene(mesh)
            # Render a simple front view
            png_data = scene.save_image(resolution=(512, 512))
            if png_data is not None:
                from io import BytesIO

                img = Image.open(BytesIO(png_data))
                test_renders[name] = np.array(img)
                print(f"  {name}: rendered {img.size}")
        except Exception as e:
            print(f"  {name}: render failed ({e})")

    if test_renders:
        print(f"Generated {len(test_renders)} test renders.")
    else:
        print("Could not generate test renders. Using placeholder analysis.")

In [ ]:
# Extract DINO features from available images
dino_features: dict[str, np.ndarray] = {}

# Use pipeline output if available, otherwise test renders
image_sources = {}
for img_path in sample_images[:5]:
    name = img_path.parent.name
    image_sources[name] = np.array(Image.open(img_path).convert("RGB"))

for name, img_array in test_renders.items():
    if name not in image_sources:
        image_sources[name] = img_array[:, :, :3] if img_array.shape[-1] == 4 else img_array

if image_sources:
    for name, img in image_sources.items():
        print(f"Extracting DINO features for {name}...")
        try:
            features = extract_dino_features_2d(img)
            dino_features[name] = features
            print(f"  Shape: {features.shape} (H_patches x W_patches x feature_dim)")
        except Exception as e:
            print(f"  Failed: {e}")

    print(f"\nDINO features extracted for {len(dino_features)} images.")
else:
    print("No images available for DINO feature extraction.")
    print("Please render some characters first or place images in output/segmentation/.")

---
## 3. Visualization & Clustering

### 3.1 Do features cluster by body part?

We use t-SNE/UMAP to project the high-dimensional features to 2D and color
each point by its ground-truth Strata region label. If features naturally
cluster by body part, they carry useful semantic information for weight prediction.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler


def assign_vertex_regions(
    mesh: trimesh.Trimesh,
    num_regions: int = 20,
) -> np.ndarray:
    """Assign each vertex a pseudo-region based on vertical position (Y-axis).

    This provides approximate ground-truth labels when actual bone weight data
    is not available from the FBX loader. The Y-axis heuristic maps:
    - Top -> head (1)
    - Upper -> chest/shoulders (2-3)
    - Middle -> spine/hips (4-5)
    - Lower -> legs/feet (12-17)

    For proper evaluation, use the Blender pipeline to extract actual bone weights.
    """
    verts = mesh.vertices
    y_min, y_max = verts[:, 1].min(), verts[:, 1].max()
    y_norm = (verts[:, 1] - y_min) / (y_max - y_min + 1e-8)

    # Simple vertical segmentation heuristic
    # Also use X-axis (lateral) to distinguish left/right
    x_center = verts[:, 0].mean()
    is_left = verts[:, 0] < x_center

    regions = np.zeros(len(verts), dtype=np.int32)

    # Head (top 10%)
    regions[y_norm > 0.90] = 1
    # Neck
    regions[(y_norm > 0.85) & (y_norm <= 0.90)] = 2
    # Chest
    regions[(y_norm > 0.70) & (y_norm <= 0.85)] = 3
    # Spine
    regions[(y_norm > 0.55) & (y_norm <= 0.70)] = 4
    # Hips
    regions[(y_norm > 0.45) & (y_norm <= 0.55)] = 5
    # Upper legs
    mask_upper_leg = (y_norm > 0.30) & (y_norm <= 0.45)
    regions[mask_upper_leg & is_left] = 12
    regions[mask_upper_leg & ~is_left] = 15
    # Lower legs
    mask_lower_leg = (y_norm > 0.15) & (y_norm <= 0.30)
    regions[mask_lower_leg & is_left] = 13
    regions[mask_lower_leg & ~is_left] = 16
    # Feet
    mask_foot = y_norm <= 0.15
    regions[mask_foot & is_left] = 14
    regions[mask_foot & ~is_left] = 17

    return regions


def plot_feature_tsne(
    features: np.ndarray,
    labels: np.ndarray,
    title: str,
    max_points: int = 5000,
) -> None:
    """Plot t-SNE of features colored by region labels."""
    # Subsample for speed
    if len(features) > max_points:
        idx = np.random.choice(len(features), max_points, replace=False)
        features = features[idx]
        labels = labels[idx]

    # Standardize
    features_scaled = StandardScaler().fit_transform(features)

    # PCA pre-reduction for speed
    if features_scaled.shape[1] > 50:
        features_scaled = PCA(n_components=50).fit_transform(features_scaled)

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    coords = tsne.fit_transform(features_scaled)

    # Plot
    _fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    unique_labels = np.unique(labels)

    for region_id in unique_labels:
        if region_id == 0:
            continue  # Skip background
        mask = labels == region_id
        region_name = REGION_NAMES.get(region_id, f"region_{region_id}")
        color = REGION_CMAP(region_id / 19.0)
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[color],
            label=region_name,
            alpha=0.6,
            s=10,
        )

    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize Diff3F features (3D vertex-level)
for name in vertex_features:
    if name not in sample_meshes:
        continue

    mesh = sample_meshes[name]
    features = vertex_features[name]
    labels = assign_vertex_regions(mesh)

    unique_vals, counts = np.unique(labels, return_counts=True)
    print(f"\n--- {name} ---")
    print(f"Vertices: {len(mesh.vertices)}, Features: {features.shape}")
    print(f"Region distribution: {dict(zip(unique_vals, counts, strict=True))}")

    plot_feature_tsne(features, labels, f"Diff3F Features: {name}")

In [ ]:
# Visualize DINO features (2D patch-level)
for name, features in dino_features.items():
    h, w, dim = features.shape
    flat_features = features.reshape(-1, dim)

    # PCA to 3 components for RGB visualization
    pca = PCA(n_components=3)
    pca_features = pca.fit_transform(flat_features)

    # Normalize to [0, 1] for display
    pca_min = pca_features.min(axis=0)
    pca_max = pca_features.max(axis=0)
    pca_features = (pca_features - pca_min) / (pca_max - pca_min + 1e-8)
    pca_image = pca_features.reshape(h, w, 3)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Original image
    if name in image_sources:
        axes[0].imshow(image_sources[name])
        axes[0].set_title(f"{name} - Original")
    axes[0].axis("off")

    # PCA feature map
    axes[1].imshow(pca_image)
    axes[1].set_title(f"{name} - DINO PCA Features")
    axes[1].axis("off")

    plt.suptitle(f"DINO Feature Visualization: {name}")
    plt.tight_layout()
    plt.show()

    print(f"  PCA explained variance: {pca.explained_variance_ratio_[:3].sum():.1%}")

### 3.2 Joint proximity analysis

Do features at joint locations (boundary between two body regions) form
distinct clusters? This is the key question: if joint features are semantically
distinct from mid-segment features, Diff3F could improve weight prediction
at deformation zones.

In [ ]:
def analyze_joint_features(
    mesh: trimesh.Trimesh,
    features: np.ndarray,
    labels: np.ndarray,
) -> dict:
    """Analyze whether features at region boundaries differ from segment interiors.

    Classifies each vertex as 'interior' (far from region boundary) or 'boundary'
    (near a vertex of a different region), then compares feature distributions.

    Returns:
        Dict with analysis results including boundary/interior feature statistics.
    """
    from scipy.spatial import KDTree

    verts = mesh.vertices
    tree = KDTree(verts)

    # For each vertex, check if any neighbor within radius has a different label
    # Use mesh edge length as scale reference
    edges = mesh.edges_unique
    edge_lengths = np.linalg.norm(verts[edges[:, 0]] - verts[edges[:, 1]], axis=1)
    median_edge = np.median(edge_lengths)
    boundary_radius = median_edge * 3  # 3x median edge length

    is_boundary = np.zeros(len(verts), dtype=bool)
    for i in range(len(verts)):
        neighbors = tree.query_ball_point(verts[i], boundary_radius)
        neighbor_labels = labels[neighbors]
        if len(np.unique(neighbor_labels)) > 1:
            is_boundary[i] = True

    # Compare feature norms and variance at boundaries vs. interiors
    boundary_features = features[is_boundary]
    interior_features = features[~is_boundary]

    results = {
        "total_vertices": len(verts),
        "boundary_count": int(is_boundary.sum()),
        "interior_count": int((~is_boundary).sum()),
        "boundary_pct": float(is_boundary.mean()),
    }

    if len(boundary_features) > 0 and len(interior_features) > 0:
        results["boundary_feature_norm_mean"] = float(
            np.linalg.norm(boundary_features, axis=1).mean()
        )
        results["interior_feature_norm_mean"] = float(
            np.linalg.norm(interior_features, axis=1).mean()
        )
        results["boundary_feature_var"] = float(boundary_features.var(axis=0).mean())
        results["interior_feature_var"] = float(interior_features.var(axis=0).mean())

    return results


# Run joint proximity analysis
for name in vertex_features:
    if name not in sample_meshes:
        continue

    mesh = sample_meshes[name]
    features = vertex_features[name]
    labels = assign_vertex_regions(mesh)

    results = analyze_joint_features(mesh, features, labels)

    print(f"\n--- {name} Joint Proximity Analysis ---")
    print(f"  Boundary vertices: {results['boundary_count']} ({results['boundary_pct']:.1%})")
    print(f"  Interior vertices: {results['interior_count']}")
    if "boundary_feature_norm_mean" in results:
        print(f"  Boundary feature norm (mean): {results['boundary_feature_norm_mean']:.4f}")
        print(f"  Interior feature norm (mean): {results['interior_feature_norm_mean']:.4f}")
        print(f"  Boundary feature variance:    {results['boundary_feature_var']:.6f}")
        print(f"  Interior feature variance:    {results['interior_feature_var']:.6f}")

### 3.3 Cross-character feature consistency

If Diff3F features are truly semantic, the same body region should have
similar features across different characters (different mesh topology,
different proportions). This is essential for generalization.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def compute_cross_character_similarity(
    all_features: dict[str, np.ndarray],
    all_labels: dict[str, np.ndarray],
    regions_to_compare: list[int] | None = None,
) -> dict[int, float]:
    """Compute average cosine similarity of features for the same region across characters.

    For each region, takes the mean feature vector from each character and computes
    pairwise cosine similarity between characters.

    Returns:
        Dict mapping region_id -> mean cosine similarity across character pairs.
    """
    if regions_to_compare is None:
        regions_to_compare = list(range(1, 20))  # All body regions

    char_names = list(all_features.keys())
    if len(char_names) < 2:
        print("Need at least 2 characters for cross-character comparison.")
        return {}

    region_similarities: dict[int, float] = {}

    for region_id in regions_to_compare:
        mean_features = []
        for name in char_names:
            mask = all_labels[name] == region_id
            if mask.sum() == 0:
                continue
            mean_feat = all_features[name][mask].mean(axis=0)
            mean_features.append(mean_feat)

        if len(mean_features) < 2:
            continue

        mean_features = np.array(mean_features)
        sim_matrix = cosine_similarity(mean_features)

        # Average off-diagonal similarities
        n = len(mean_features)
        off_diag = sim_matrix[np.triu_indices(n, k=1)]
        region_similarities[region_id] = float(off_diag.mean())

    return region_similarities


# Compute cross-character consistency
if len(vertex_features) >= 2:
    all_labels = {
        name: assign_vertex_regions(sample_meshes[name])
        for name in vertex_features
        if name in sample_meshes
    }

    similarities = compute_cross_character_similarity(vertex_features, all_labels)

    print("\nCross-Character Feature Similarity by Region:")
    print(f"{'Region':<20} {'Cosine Sim':>12}")
    print("-" * 35)
    for region_id, sim in sorted(similarities.items()):
        region_name = REGION_NAMES.get(region_id, f"region_{region_id}")
        quality = "HIGH" if sim > 0.7 else ("MEDIUM" if sim > 0.4 else "LOW")
        print(f"  {region_name:<18} {sim:>8.4f}  [{quality}]")

    if similarities:
        mean_sim = np.mean(list(similarities.values()))
        print(f"\n  Overall mean: {mean_sim:.4f}")
else:
    print("Need at least 2 characters with features for cross-character analysis.")

---
## 4. Evaluation

### 4.1 Region classification accuracy

Train a simple classifier (k-NN or logistic regression) to predict body region
from features. Compare:
1. **Geometric only** — vertex position (x, y, z) normalized
2. **Diff3F only** — 2048-dim semantic features
3. **Combined** — position + Diff3F features

If Diff3F features improve classification, they carry useful signal for
weight prediction.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline


def evaluate_region_classification(
    mesh: trimesh.Trimesh,
    features: np.ndarray,
    labels: np.ndarray,
    name: str,
) -> dict[str, float]:
    """Compare region classification accuracy across feature sets."""
    # Filter out background (label 0)
    valid = labels > 0
    labels_valid = labels[valid]
    features_valid = features[valid]
    positions_valid = mesh.vertices[valid]

    # Normalize positions to [0, 1]
    pos_min = positions_valid.min(axis=0)
    pos_max = positions_valid.max(axis=0)
    positions_norm = (positions_valid - pos_min) / (pos_max - pos_min + 1e-8)

    results = {}

    # Feature sets to compare
    feature_sets = {
        "geometric_only": positions_norm,
        "diff3f_only": features_valid,
        "combined": np.hstack([positions_norm, features_valid]),
    }

    for feat_name, X in feature_sets.items():
        # PCA to reduce dimensionality for classifier stability
        n_components = min(50, X.shape[1], X.shape[0] - 1)
        pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=n_components)),
                ("clf", KNeighborsClassifier(n_neighbors=5)),
            ]
        )

        # 5-fold cross-validation
        n_samples = min(len(labels_valid), 5000)  # Cap for speed
        if n_samples < len(labels_valid):
            idx = np.random.choice(len(labels_valid), n_samples, replace=False)
            X_sub, y_sub = X[idx], labels_valid[idx]
        else:
            X_sub, y_sub = X, labels_valid

        scores = cross_val_score(pipe, X_sub, y_sub, cv=5, scoring="accuracy")
        results[feat_name] = float(scores.mean())

    return results


# Run classification experiment
print("Region Classification Accuracy (5-fold CV):\n")
all_results: dict[str, dict[str, float]] = {}

for name in vertex_features:
    if name not in sample_meshes:
        continue

    mesh = sample_meshes[name]
    features = vertex_features[name]
    labels = assign_vertex_regions(mesh)

    results = evaluate_region_classification(mesh, features, labels, name)
    all_results[name] = results

    print(f"  {name}:")
    for feat_name, acc in results.items():
        print(f"    {feat_name:<20} {acc:.1%}")
    print()

# Summary
if all_results:
    print("\nMean Accuracy Across Characters:")
    for feat_name in ["geometric_only", "diff3f_only", "combined"]:
        accs = [r[feat_name] for r in all_results.values() if feat_name in r]
        if accs:
            improvement = ""
            if feat_name in ("diff3f_only", "combined"):
                geo_accs = [r["geometric_only"] for r in all_results.values()]
                delta = np.mean(accs) - np.mean(geo_accs)
                improvement = f"  ({'+' if delta > 0 else ''}{delta:.1%} vs geometric)"
            print(f"  {feat_name:<20} {np.mean(accs):.1%}{improvement}")

### 4.2 Weight prediction relevance

Strata's weight prediction outputs per-vertex skinning weights (a blend of
influences from nearby bones). We evaluate whether Diff3F features correlate
with actual bone weight distributions — i.e., do vertices with similar Diff3F
features also have similar weight profiles?

In [ ]:
def compute_weight_feature_correlation(
    features: np.ndarray,
    labels: np.ndarray,
) -> dict:
    """Measure how well features predict intra-region vs. inter-region distances.

    If features are semantically meaningful, vertices within the same region should
    have higher feature similarity than vertices across regions (analogous to how
    bone weights are similar within a region).

    Returns:
        Dict with intra/inter region similarity ratios.
    """
    valid = labels > 0
    features_valid = features[valid]
    labels_valid = labels[valid]

    # Subsample for tractable pairwise computation
    n = min(2000, len(features_valid))
    idx = np.random.choice(len(features_valid), n, replace=False)
    features_sub = features_valid[idx]
    labels_sub = labels_valid[idx]

    # Normalize features
    features_sub = StandardScaler().fit_transform(features_sub)

    # Compute pairwise cosine similarity
    sim_matrix = cosine_similarity(features_sub)

    # Separate intra-region and inter-region pairs
    same_region = labels_sub[:, None] == labels_sub[None, :]
    upper_tri = np.triu_indices(n, k=1)

    intra_sims = sim_matrix[upper_tri][same_region[upper_tri]]
    inter_sims = sim_matrix[upper_tri][~same_region[upper_tri]]

    return {
        "intra_region_sim_mean": float(intra_sims.mean()) if len(intra_sims) > 0 else 0.0,
        "inter_region_sim_mean": float(inter_sims.mean()) if len(inter_sims) > 0 else 0.0,
        "intra_region_sim_std": float(intra_sims.std()) if len(intra_sims) > 0 else 0.0,
        "inter_region_sim_std": float(inter_sims.std()) if len(inter_sims) > 0 else 0.0,
        "separation_ratio": (
            float(intra_sims.mean() / (inter_sims.mean() + 1e-8))
            if len(intra_sims) > 0 and len(inter_sims) > 0
            else 0.0
        ),
    }


# Analyze weight-feature correlation
print("Weight-Feature Correlation Analysis:\n")

for name in vertex_features:
    if name not in sample_meshes:
        continue

    mesh = sample_meshes[name]
    features = vertex_features[name]
    labels = assign_vertex_regions(mesh)

    corr = compute_weight_feature_correlation(features, labels)

    print(f"  {name}:")
    print(
        f"    Intra-region similarity: {corr['intra_region_sim_mean']:.4f} +/- {corr['intra_region_sim_std']:.4f}"
    )
    print(
        f"    Inter-region similarity: {corr['inter_region_sim_mean']:.4f} +/- {corr['inter_region_sim_std']:.4f}"
    )
    print(f"    Separation ratio:        {corr['separation_ratio']:.2f}x")
    quality = (
        "GOOD"
        if corr["separation_ratio"] > 1.5
        else ("MODERATE" if corr["separation_ratio"] > 1.1 else "POOR")
    )
    print(f"    Assessment: {quality}")
    print()

---
## 5. Conclusion & Recommendation

In [ ]:
print("=" * 60)
print("DIFF3F EXPLORATION SUMMARY")
print("=" * 60)

print("""
Findings:
---------
(Fill in after running the above experiments)

1. Feature clustering by body part:
   [ ] Strong clustering — features clearly separate body regions
   [ ] Moderate clustering — some separation but noisy
   [ ] Weak clustering — no meaningful body-part grouping

2. Joint proximity distinction:
   [ ] Joint/boundary features are distinct from mid-segment
   [ ] No clear boundary vs. interior distinction

3. Cross-character consistency:
   [ ] Same region -> similar features across characters
   [ ] Features are character-specific (don't generalize)

4. Region classification improvement:
   Geometric-only accuracy:  ____%
   Diff3F-only accuracy:     ____%
   Combined accuracy:        ____%

5. Weight prediction relevance:
   Separation ratio:         ____x

Recommendation:
---------------
[ ] INTEGRATE — Diff3F features provide significant improvement.
    Next step: File follow-up issue to add Diff3F features to weight
    prediction pipeline.

[ ] SKIP — Features don't provide enough improvement to justify the
    computational cost (GPU, model weights, extraction time).

[ ] EXPLORE FURTHER — Results are inconclusive. Consider:
    - Testing with actual bone weights (not heuristic labels)
    - Using DINOv2 features on 2D images as a lighter alternative
    - Fine-tuning on Strata-specific data

Practical Considerations:
-------------------------
- Diff3F requires 3D meshes at feature-extraction time
  (fine for training data generation, not for 2D inference)
- DINOv2 features on 2D images may be a more practical path
  for inference-time weight prediction
- Extraction time: ~2-5 min per mesh on a modern GPU
- Model weights: ~5GB (Stable Diffusion + ControlNet + DINO)
""")

In [ ]:
# If results are promising, create a follow-up issue template
FOLLOWUP_TEMPLATE = """
## Follow-up: Integrate Diff3F/DINO Features into Weight Prediction

Based on exploration in docs/research/diff3f_exploration.ipynb:

### Proposed Changes
1. Add DINO feature extraction to `pipeline/weight_extractor.py`
2. Extend per-vertex weight data with semantic feature vectors
3. Train weight prediction model with combined geometric + semantic features

### Key Metrics from Exploration
- Region classification: geometric=___% vs combined=___% (+___pp)
- Separation ratio: ___x
- Cross-character consistency: ___

### Implementation Notes
- Use DINOv2 (2D) features for inference, Diff3F (3D) for training data
- Feature dim: 768 (DINOv2-base) or 384 (DINOv2-small) per patch
- GPU required for feature extraction (~0.5s per image with DINOv2)
"""

print("Follow-up issue template (use if results are promising):")
print(FOLLOWUP_TEMPLATE)